# 🎓 Define Tu Futuro — Análisis Exploratorio de Datos (EDA) y Modelamiento Estadístico DEMRE

**Proyecto Final del Curso de Código y Programación**  
**Samsung Innovation Campus Chile 2026 – Cohort 2**

---

## 📌 Descripción del Notebook

Este notebook documenta el proceso completo de análisis de datos, ingeniería de variables, agrupamiento estadístico por percentiles y entrenamiento de modelos de Machine Learning utilizando datos del proceso de admisión universitaria chileno (**DEMRE**).

### 🎯 Objetivos:
1. **Carga y Limpieza de Datos**: Procesar missing values, corregir encodings de texto y codificar variables categóricas.
2. **Ingeniería de Características**: Calcular el puntaje ponderado estimado según reglas heurísticas por área del conocimiento (Ingeniería, Salud, Humanidades, Comercial, etc.).
3. **Análisis Exploratorio de Datos (EDA)**: Estudiar distribuciones de puntajes, correlaciones socioeconómicas, brechas por tipo de colegio y previsión de salud.
4. **Agrupamiento por Carrera e Institución**: Calcular percentiles históricos (P10, P25, P50, P75, P90) y promedios por carrera.
5. **Modelamiento Predictivo**: Entrenar un clasificador **Random Forest** para estimar la probabilidad de admisión y evaluar su rendimiento.
6. **Sistema de Predicción Calibrado**: Combinar la predicción del modelo con interpolación lineal de percentiles empíricos y simular un informe de orientación vocacional asistido por IA (HuggingFace Qwen 2.5).


## 🛠️ 1. Importación de Librerías y Configuración General

Importamos las herramientas fundamentales de Python para procesamiento de datos (`pandas`, `numpy`), visualización (`matplotlib`, `seaborn`), modelamiento (`scikit-learn`) y serialización (`joblib`).


In [ ]:
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score
)

# Configuración de estilos visuales
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas exitosamente.")


## 🧹 2. Carga y Preprocesamiento de Datos DEMRE

Cargamos el dataset histórico. El script contempla la carga desde el archivo CSV original (`DEMRE_1.csv`) o desde la muestra guardada en los artefactos del proyecto (`models/demre_sample_eda.joblib`).


In [ ]:
# Ubicación de archivos
base_dir = os.getcwd()
sample_path = os.path.join(base_dir, 'models', 'demre_sample_eda.joblib')
csv_path = os.path.join(base_dir, 'DEMRE_1.csv')

def clean_text(text):
    if not isinstance(text, str):
        return text
    replacements = {
        'Ingeniera': 'Ingeniería', 'Enfermera': 'Enfermería', 'Psicologa': 'Psicología',
        'Tecnologa': 'Tecnología', 'Mdica': 'Médica', 'Pedagoga': 'Pedagogía',
        'Educacin': 'Educación', 'Comunicacin': 'Comunicación', 'Biologa': 'Biología',
        'Qumica': 'Química', 'Geografa': 'Geografía', 'Odontologa': 'Odontología',
        'Kinesiologa': 'Kinesiología', 'Sociologa': 'Sociología', 'Filosofa': 'Filosofía',
        'Comn': 'Común', 'Fsica': 'Física', 'Agronoma': 'Agronomía',
        'Construccin': 'Construcción', 'Auditora': 'Auditoría', 'Diseo': 'Diseño'
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text.strip()

# Carga de datos con mecanismo de fallback
if os.path.exists(sample_path):
    print(f"Cargando muestra procesada desde: {sample_path}")
    df = joblib.load(sample_path)
elif os.path.exists(csv_path):
    print(f"Cargando CSV desde: {csv_path}")
    df = pd.read_csv(csv_path, sep=';', encoding='latin-1')
    if len(df) > 80000:
        df = df.sample(80000, random_state=42).copy()
else:
    raise FileNotFoundError("No se encontró el dataset DEMRE ni la muestra en models/.")

print(f"Dataset cargado correctamente. Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas.")
df.head(3)


### 2.1 Imputación y Codificación de Variables Sociodemográficas

Se realizan las siguientes transformaciones:
1. **Limpieza de Texto**: Corrección de codificaciones de caracteres en nombres de carrera e institución.
2. **Imputación de Puntajes**: Reemplazo de missing values en puntajes con el valor neutro de **550.0 pts**.
3. **Imputación de Cuantil**: Imputación del cuantil de ingreso faltante con el valor mediano **3.0**.
4. **Variables Binarias**: Codificación de tipo de colegio (particular pagado, subvencionado), trabajo remunerado y sexo femenino.


In [ ]:
# Tratamiento de nulos en puntajes académicos
score_cols = ['ptje_nem', 'ptje_leng', 'ptje_mate', 'ptje_hycs', 'ptje_cien']
for col in score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(550.0)

# Imputación de cuantil socioeconómico
if 'cuantil_ingreso_bruto_fam' in df.columns:
    df['cuantil_ingreso_bruto_fam'] = pd.to_numeric(df['cuantil_ingreso_bruto_fam'], errors='coerce').fillna(3.0)

# Codificación de variables categóricas explicativas
if 'nombre_dependencia_establecimiento' in df.columns:
    df['colegio_particular_pagado'] = (df['nombre_dependencia_establecimiento'].astype(str).str.contains('Pagado', case=False, na=False)).astype(int)
    df['colegio_subvencionado'] = (df['nombre_dependencia_establecimiento'].astype(str).str.contains('Subvencionado', case=False, na=False)).astype(int)

if 'descripcion_trabajo_remunerado' in df.columns:
    df['trabaja_remunerado'] = (df['descripcion_trabajo_remunerado'].astype(str).str.contains('Sí|Si', case=False, na=False)).astype(int)

if 'nombre_sexo' in df.columns:
    df['es_femenino'] = (df['nombre_sexo'].astype(str).str.contains('Femenino', case=False, na=False)).astype(int)

# Prueba específica máxima (Historia vs Ciencias)
df['ptje_especifica_max'] = df[['ptje_hycs', 'ptje_cien']].max(axis=1)

print("✅ Variables numéricas y binarias procesadas correctamente.")


### 2.2 Cálculo del Puntaje Ponderado Estimado

El proyecto aplica ponderaciones simuladas basadas en palabras clave contenidas en el nombre de cada programa académico:

- **Ingeniería / Informática / Física**: NEM 20%, Lenguaje 15%, Matemática 45%, Específica 20%
- **Salud / Medicina / Odontología**: NEM 25%, Lenguaje 15%, Matemática 25%, Ciencias 35%
- **Derecho / Psicología / Periodismo**: NEM 20%, Lenguaje 35%, Matemática 15%, Historia 30%
- **Comercial / Economía / Auditoría**: NEM 20%, Lenguaje 25%, Matemática 40%, Específica 15%
- **Otras disciplinas**: NEM 20%, Lenguaje 25%, Matemática 35%, Específica 20%


In [ ]:
nem = df['ptje_nem']
leng = df['ptje_leng']
mate = df['ptje_mate']
esp = df['ptje_especifica_max']

if 'nombre_carrera_normalizacion' in df.columns:
    carreras = df['nombre_carrera_normalizacion'].astype(str).str.lower()
else:
    carreras = pd.Series("", index=df.index)

is_ing = carreras.str.contains('ingenier|matemat|fisic|inform', regex=True, na=False)
is_salud = carreras.str.contains('medicin|enfermer|salud|kinesiolog|odontol', regex=True, na=False)
is_hum = carreras.str.contains('derecho|periodis|histori|psicolog', regex=True, na=False)
is_com = carreras.str.contains('comercial|econom|auditor', regex=True, na=False)

df['puntaje_ponderado_estimado'] = (
    np.where(is_ing, nem * 0.20 + leng * 0.15 + mate * 0.45 + esp * 0.20,
    np.where(is_salud, nem * 0.25 + leng * 0.15 + mate * 0.25 + df['ptje_cien'] * 0.35,
    np.where(is_hum, nem * 0.20 + leng * 0.35 + mate * 0.15 + df['ptje_hycs'] * 0.30,
    np.where(is_com, nem * 0.20 + leng * 0.25 + mate * 0.40 + esp * 0.15,
    nem * 0.20 + leng * 0.25 + mate * 0.35 + esp * 0.20))))
)

print(f"Resumen Puntaje Ponderado Estimado:\n{df['puntaje_ponderado_estimado'].describe()}")


## 📊 3. Análisis Exploratorio de Datos (EDA)

En esta sección exploramos las distribuciones de las variables académicas y socioeconómicas para responder las preguntas de investigación.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Distribuciones
sns.histplot(df['ptje_nem'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribución de Puntajes NEM')

sns.histplot(df['ptje_leng'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribución de Lenguaje / Comp. Lectora')

sns.histplot(df['ptje_mate'], kde=True, ax=axes[0, 2], color='lightgreen')
axes[0, 2].set_title('Distribución de Matemática (M1)')

sns.histplot(df['ptje_especifica_max'], kde=True, ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Distribución de Prueba Específica Máx.')

sns.histplot(df['puntaje_ponderado_estimado'], kde=True, ax=axes[1, 1], color='orange')
axes[1, 1].set_title('Distribución del Ponderado Estimado')

if 'cuantil_ingreso_bruto_fam' in df.columns:
    sns.countplot(data=df, x='cuantil_ingreso_bruto_fam', ax=axes[1, 2], palette='viridis')
    axes[1, 2].set_title('Distribución por Cuantil de Ingreso Familiar')

plt.tight_layout()
plt.show()


### 3.1 Matriz de Correlación de Pearson

Analizamos la intensidad de la relación lineal entre puntajes académicos y el cuantil de ingreso socioeconómico.


In [ ]:
corr_cols = [
    'ptje_nem', 'ptje_leng', 'ptje_mate', 'ptje_especifica_max',
    'puntaje_ponderado_estimado', 'cuantil_ingreso_bruto_fam'
]
available_corr_cols = [c for c in corr_cols if c in df.columns]

plt.figure(figsize=(9, 6))
corr_matrix = df[available_corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Matriz de Correlación de Pearson — Variables Académicas y Socioeconómicas')
plt.show()


### 3.2 Brecha de Puntajes según Dependencia del Establecimiento Educacional


In [ ]:
if 'nombre_dependencia_establecimiento' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=df,
        x='nombre_dependencia_establecimiento',
        y='puntaje_ponderado_estimado',
        palette='Set2'
    )
    plt.xticks(rotation=15)
    plt.title('Comparación de Puntaje Ponderado Estimado según Tipo de Colegio')
    plt.xlabel('Dependencia Escolar')
    plt.ylabel('Puntaje Ponderado Estimado')
    plt.show()


## 📈 4. Agrupamiento Estadístico y Percentiles por Carrera

Agrupamos los registros por combinación de **Institución + Carrera** y calculamos los percentiles históricos (P10, P25, P50, P75, P90). El percentil **P25** se utiliza como proxy del puntaje de corte estimado.


In [ ]:
def generate_career_statistics(data):
    grp = data.groupby(['nombre_institucion_educacion_superior', 'nombre_carrera_normalizacion']).agg(
        total_postulaciones=('ptje_nem', 'count'),
        nem_prom=('ptje_nem', 'mean'),
        leng_prom=('ptje_leng', 'mean'),
        mate_prom=('ptje_mate', 'mean'),
        hycs_prom=('ptje_hycs', 'mean'),
        cien_prom=('ptje_cien', 'mean'),
        esp_max_prom=('ptje_especifica_max', 'mean'),
        ponderado_prom=('puntaje_ponderado_estimado', 'mean'),
        cuantil_ingreso_prom=('cuantil_ingreso_bruto_fam', 'mean') if 'cuantil_ingreso_bruto_fam' in data.columns else ('ptje_nem', lambda x: 3.0),
        ponderado_p10=('puntaje_ponderado_estimado', lambda x: float(np.percentile(x, 10))),
        ponderado_p25=('puntaje_ponderado_estimado', lambda x: float(np.percentile(x, 25))),
        ponderado_p50=('puntaje_ponderado_estimado', lambda x: float(np.percentile(x, 50))),
        ponderado_p75=('puntaje_ponderado_estimado', lambda x: float(np.percentile(x, 75))),
        ponderado_p90=('puntaje_ponderado_estimado', lambda x: float(np.percentile(x, 90)))
    ).reset_index()
    
    grp = grp[grp['total_postulaciones'] >= 3].copy()
    return grp

stats_df = generate_career_statistics(df)
print(f"Total de combinaciones Carrera-Institución identificadas: {len(stats_df):,}")
stats_df.head(5)


### 4.1 Top 10 Carreras e Instituciones con Mayor Puntaje de Corte (P25)


In [ ]:
top_cuts = stats_df.sort_values(by='ponderado_p25', ascending=False).head(10)
top_cuts[['nombre_institucion_educacion_superior', 'nombre_carrera_normalizacion', 'total_postulaciones', 'ponderado_p25', 'ponderado_p50']]


## 🤖 5. Entrenamiento del Modelo de Machine Learning (Random Forest)

### 5.1 Etiqueta Objetivo Estimada
Definimos la etiqueta binaria de entrenamiento:
- `es_admitido_estimado = 1` si el ponderado estimado es mayor o igual al P25 histórico de esa carrera.
- `es_admitido_estimado = 0` en caso contrario.

### 5.2 Algoritmo y Variables
Entrenamos un **RandomForestClassifier** (`n_estimators=70`, `max_depth=10`, `random_state=42`) utilizando las variables académicas y sociodemográficas.


In [ ]:
# Join dataset con estadísticas para obtener P25 y P50 por carrera
df_merged = df.merge(
    stats_df[['nombre_institucion_educacion_superior', 'nombre_carrera_normalizacion', 'ponderado_p50', 'ponderado_p25']],
    on=['nombre_institucion_educacion_superior', 'nombre_carrera_normalizacion'],
    how='inner'
)

# Definición de etiqueta estimada
df_merged['es_admitido_estimado'] = (df_merged['puntaje_ponderado_estimado'] >= df_merged['ponderado_p25']).astype(int)

features = [
    'ptje_nem', 'ptje_leng', 'ptje_mate', 'ptje_especifica_max', 
    'puntaje_ponderado_estimado', 'ponderado_p50',
    'cuantil_ingreso_bruto_fam', 'colegio_particular_pagado', 
    'colegio_subvencionado', 'trabaja_remunerado', 'es_femenino'
]

available_features = [f for f in features if f in df_merged.columns]

X = df_merged[available_features].dropna()
y = df_merged.loc[X.index, 'es_admitido_estimado']

# Split de datos (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=70, max_depth=10, random_state=42, n_jobs=-1)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"✅ Modelo Random Forest Entrenado.")
print(f"Exactitud (Accuracy): {acc * 100:.2f}%")
print(f"Área Bajo la Curva (ROC-AUC): {roc_auc:.4f}\n")
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred))


### 5.3 Evaluación de Rendimiento: Matriz de Confusión y Curva ROC


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusión')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Valor Real')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[1].set_title('Curva ROC')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


### 5.4 Importancia de las Variables (Feature Importance)

Analizamos cuáles características influyen en mayor medida en la predicción del Random Forest.


In [ ]:
importances = pd.Series(clf.feature_importances_, index=available_features).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='teal')
plt.title('Importancia de Variables en el Modelo Random Forest')
plt.xlabel('Importancia Relativa (Gini Importance)')
plt.show()


## 🔮 6. Simulación de Predicción Calibrada e Informe IA

### 6.1 Predicción Combinada (50% ML + 50% Interpolación Empírica)
Probamos la función de predicción con un perfil de estudiante hipotético y generamos el prompt vocacional adaptado (`construir_prompt`).


In [ ]:
from hugging_face.prompts import construir_prompt

# Perfil hipotético de prueba
perfil_ejemplo = {
    "sexo": "Femenino",
    "nacionalidad": "Chilena",
    "region": "Región Metropolitana",
    "dependencia": "Particular Subvencionado",
    "rama": "Humanista Científico",
    "cuantil": 5,
    "trabajo": "No",
    "financiamiento": "Beca de Excelencia Académica",
    "salud": "FONASA",
    "convivencia": "Con ambos padres",
    "jefe_hogar": "Madre",
    "carrera": "Ingeniería Civil Informática",
    "institucion": "Universidad de Chile"
}

puntajes_ejemplo = {
    "nem": 680,
    "lenguaje": 650,
    "matematica": 670,
    "tipo_electiva": "Ciencias",
    "puntaje_electiva": 630
}

prediccion_ejemplo = {
    "probabilidad": 78.4,
    "etiqueta": "Alta Probabilidad de Ingreso",
    "user_ponderado": 662.5,
    "corte_p25": 625.0,
    "promedio_p50": 655.0,
    "promedios_carrera": {
        "NEM": 650.0,
        "Lenguaje": 630.0,
        "Matemáticas": 660.0,
        "Historia": 550.0,
        "Ciencias": 620.0
    },
    "factores": [
        "⬆️ Cuantil socioeconómico superior a la media de la carrera (+3% ajuste de retención histórica)"
    ]
}

contexto_ejemplo = {
    "target_affinity": 62.5,
    "target_rows": 240,
    "exact_matches": 5,
    "nearest_ponderado": 658.0
}

distribucion_areas_ejemplo = {
    "Ingeniería": 70.0,
    "Técnica": 10.0,
    "Otra": 20.0
}

carreras_alternativas_ejemplo = [
    {
        "carrera": "Ingeniería Civil Industrial",
        "institucion": "Universidad de Santiago de Chile",
        "probabilidad": 84.0,
        "corte_estimado": 615.0,
        "afinidad_contextual": 68.0
    },
    {
        "carrera": "Ingeniería Comercial",
        "institucion": "Pontificia Universidad Católica de Chile",
        "probabilidad": 72.0,
        "corte_estimado": 640.0,
        "afinidad_contextual": 58.0
    }
]

# Generación del prompt adaptado para HuggingFace
prompt_generado = construir_prompt(
    perfil=perfil_ejemplo,
    puntajes=puntajes_ejemplo,
    prediccion=prediccion_ejemplo,
    contexto=contexto_ejemplo,
    distribucion_areas=distribucion_areas_ejemplo,
    carreras_alternativas=carreras_alternativas_ejemplo
)

print("=== PROMPT GENERADO PARA LA IA (HuggingFace / Qwen 2.5) ===")
print(prompt_generado[:1200] + "\n... [TRUNCADO PARA PRESENTACIÓN] ...")


## 🏁 7. Conclusiones y Próximas Mejoras

### 📌 Conclusiones Clave:
1. **Ponderado Estimado e Importancia**: Las variables académicas (`puntaje_ponderado_estimado`, `ponderado_p50`, `ptje_mate`, `ptje_nem`) son los predictores con mayor importancia Gini en la determinación de admisión estimada.
2. **Calibración Híbrida**: La combinación al 50% entre la probabilidad del Random Forest y la posición lineal dentro de percentiles (P10-P90) otorga estabilidad a la probabilidad mostrada al usuario.
3. **Integración con IA**: El módulo `hugging_face` transforma esta analítica tabular densa en un informe de orientación vocacional redactado en lenguaje natural, presentado dinámicamente mediante el sistema Bento Grid en Streamlit.

---
**Samsung Innovation Campus Chile 2026 – Define Tu Futuro**
